# TMDB Box Office Prediction v2

[Kaggle competition](https://www.kaggle.com/c/tmdb-box-office-prediction) — metric: RMSLE. Log-target gradient-boosting ensemble (LightGBM + XGBoost + CatBoost).

In [ ]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd
from src.features import build_features
from src.models import make_models, train_oof, blend
from src.evaluate import rmsle, results_table

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print(train.shape, test.shape)
train.head()

## Feature engineering
Train on `log1p(revenue)` (the original trained on raw revenue — the core bug). Mine counts/flags from the JSON columns the original discarded.

In [ ]:
X, y, X_test, feats = build_features(train, test)
print(f'{len(feats)} features')

## Models & 5-fold cross-validation

In [ ]:
scores, oof_dict, test_dict = {}, {}, {}
for name in make_models().keys():
    oof, tp, cv = train_oof(name, X, y, X_test, folds=5)
    scores[name], oof_dict[name], test_dict[name] = cv, oof, tp
    print(name, round(cv, 5))

In [ ]:
blended_oof, blended_test, weights = blend(oof_dict, y, test_dict)
scores['ensemble'] = rmsle(np.expm1(y.values), np.expm1(blended_oof))
print('weights', weights)
print(results_table(scores))

## Results

In [ ]:
preds = np.clip(np.expm1(blended_test), 0, None)
pd.DataFrame({'id': test['id'].astype(int), 'revenue': preds}).to_csv('../data/submission.csv', index=False)
print('submission written')